In [ ]:
!pip install ninja

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 422.9/422.9 kB 5.2 MB/s eta 0:00:00


In [ ]:
!git clone https://github.com/PeikeLi/Self-Correction-Human-Parsing
%cd Self-Correction-Human-Parsing
!mkdir checkpoints
!mkdir inputs
!mkdir outputs

Cloning into 'Self-Correction-Human-Parsing'...
remote: Enumerating objects: 722, done.
remote: Counting objects: 100% (175/175), done.
remote: Compressing objects: 100% (110/110), done.
remote: Total 722 (delta 74), reused 64 (delta 64), pack-reused 547 (from 1)
Receiving objects: 100% (722/722), 3.88 MiB | 14.14 MiB/s, done.
Resolving deltas: 100% (150/150), done.
/content/Self-Correction-Human-Parsing


In [ ]:
dataset = 'lip'

In [ ]:
import gdown

if dataset == 'lip':
    url = 'https://drive.google.com/uc?id=1k4dllHpu0bdx38J7H28rVVLpU-kOHmnH'
elif dataset == 'atr':
    url = 'https://drive.google.com/uc?id=1ruJg4lqR_jgQPj-9K0PP-L2vJERYOxLP'
elif dataset == 'pascal':
    url = 'https://drive.google.com/uc?id=1E5YwNKW2VOEayK9mWCS3Kpsxf-3z04ZE'

output = 'checkpoints/final.pth'
gdown.download(url, output, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1k4dllHpu0bdx38J7H28rVVLpU-kOHmnH
From (redirected): https://drive.google.com/uc?id=1k4dllHpu0bdx38J7H28rVVLpU-kOHmnH&confirm=t&uuid=09b342ce-eb7e-4373-90eb-7f139a8b30af
To: /content/Self-Correction-Human-Parsing/checkpoints/final.pth
100%|██████████| 267M/267M [00:05<00:00, 51.1MB/s]


'checkpoints/final.pth'

In [ ]:
!python3 simple_extractor.py --dataset 'lip' --model-restore 'checkpoints/final.pth' --input-dir 'inputs' --output-dir 'outputs'

In [ ]:
#binary mask for source head and body
import numpy as np
from PIL import Image
import os

def create_binary_masks(parsing_result):
    """
    Create binary masks for head, body, and neck from LIP parsing result
    LIP Labels:
    'Background' (0), 'Hat' (1), 'Hair' (2), 'Glove' (3), 'Sunglasses' (4),
    'Upper-clothes' (5), 'Dress' (6), 'Coat' (7), 'Socks' (8), 'Pants' (9),
    'Jumpsuits' (10), 'Scarf' (11), 'Skirt' (12), 'Face' (13),
    'Left-arm' (14), 'Right-arm' (15), 'Left-leg' (16), 'Right-leg' (17),
    'Left-shoe' (18), 'Right-shoe' (19)
    """
    # Convert parsing result to numpy array if it's a PIL Image
    if isinstance(parsing_result, Image.Image):
        parsing_array = np.array(parsing_result)
    else:
        parsing_array = parsing_result
        
    # Initialize empty masks
    head_mask = np.zeros_like(parsing_array, dtype=np.uint8)
    body_mask = np.zeros_like(parsing_array, dtype=np.uint8)
    neck_mask = np.zeros_like(parsing_array, dtype=np.uint8)
    
    # Define labels for head and body
    head_labels = {1, 2, 4, 13}  # Hat, Hair, Sunglasses, Face
    upper_body_labels = {5, 6, 7}  # Upper-clothes, Dress, Coat
    body_labels = {5, 6, 7, 8, 9, 10, 12, 14, 15, 16, 17, 18, 19}  # All body parts
    
    # Create basic masks
    head_mask = np.isin(parsing_array, list(head_labels))
    body_mask = np.isin(parsing_array, list(body_labels))
    upper_body = np.isin(parsing_array, list(upper_body_labels))
    face = parsing_array == 13  # Face only
    
    # Create neck mask by finding the region between face and upper clothes
    face_bottom = np.zeros_like(face)
    upper_body_top = np.zeros_like(upper_body)
    
    # For each column, find the lowest face pixel and highest upper body pixel
    for col in range(face.shape[1]):
        face_pixels = np.where(face[:, col])[0]
        upper_pixels = np.where(upper_body[:, col])[0]
        
        if len(face_pixels) > 0 and len(upper_pixels) > 0:
            face_bottom_pixel = np.max(face_pixels)
            upper_top_pixel = np.min(upper_pixels)
            
            # Mark the region between face and upper body as neck
            if upper_top_pixel > face_bottom_pixel:
                neck_mask[face_bottom_pixel:upper_top_pixel, col] = 1
    
    # Convert to uint8 with 255 for white
    head_mask = head_mask.astype(np.uint8) * 255
    body_mask = body_mask.astype(np.uint8) * 255
    neck_mask = neck_mask.astype(np.uint8) * 255
    
    return head_mask, body_mask, neck_mask

# Add this after your semantic parsing command
output_dir = 'outputs'
binary_masks_dir = os.path.join(output_dir, 'binary_masks')
os.makedirs(binary_masks_dir, exist_ok=True)

# Process all parsed images in the outputs directory
for filename in os.listdir(output_dir):
    if filename.endswith('.png'):
        # Load the parsing result
        parsing_result = Image.open(os.path.join(output_dir, filename))
        
        # Generate binary masks
        head_mask, body_mask, neck_mask = create_binary_masks(parsing_result)
        
        # Save masks
        base_name = os.path.splitext(filename)[0]
        head_mask_img = Image.fromarray(head_mask)
        body_mask_img = Image.fromarray(body_mask)
        neck_mask_img = Image.fromarray(neck_mask)
        
        head_mask_img.save(os.path.join(binary_masks_dir, f'{base_name}_head_mask.png'))
        body_mask_img.save(os.path.join(binary_masks_dir, f'{base_name}_body_mask.png'))
        neck_mask_img.save(os.path.join(binary_masks_dir, f'{base_name}_neck_mask.png'))

In [ ]:
import cv2
import matplotlib.pyplot as plt

def blend_semantic_layouts(source_head_semantic, source_body_semantic, head_mask, body_mask, target_size=(480, 480)):
    """
    Blend semantic layouts using binary masks with alignment
    """
    # First resize all inputs to the same size
    source_head_semantic = cv2.resize(source_head_semantic, target_size[::-1], interpolation=cv2.INTER_NEAREST)
    source_body_semantic = cv2.resize(source_body_semantic, target_size[::-1], interpolation=cv2.INTER_NEAREST)
    head_mask = cv2.resize(head_mask, target_size[::-1], interpolation=cv2.INTER_NEAREST)
    body_mask = cv2.resize(body_mask, target_size[::-1], interpolation=cv2.INTER_NEAREST)
    
    # Debug print mask values
    print("Head mask range:", np.min(head_mask), np.max(head_mask))
    print("Body mask range:", np.min(body_mask), np.max(body_mask))
    
    # Find centers safely
    head_indices = np.where(head_mask > 127)
    body_indices = np.where(body_mask > 127)
    
    if len(head_indices[1]) == 0 or len(body_indices[1]) == 0:
        print("Warning: Empty mask detected, falling back to simple blending")
        # Fall back to simple blending if masks are empty
        head_mask_bool = head_mask > 127
        body_mask_bool = body_mask > 127
        
        blended_layout = np.zeros_like(source_head_semantic)
        blended_layout[head_mask_bool] = source_head_semantic[head_mask_bool]
        blended_layout[body_mask_bool] = source_body_semantic[body_mask_bool]
        
        return blended_layout
    
    # Calculate centers
    head_center = np.mean(head_indices[1])
    body_center = np.mean(body_indices[1])
    
    print(f"Head center: {head_center}")
    print(f"Body center: {body_center}")
    
    # Calculate horizontal offset
    horizontal_offset = int(round(body_center - head_center))
    print(f"Horizontal offset: {horizontal_offset}")
    
    # Apply horizontal shift if needed
    if horizontal_offset != 0:
        # Shift head semantic and mask
        if horizontal_offset > 0:
            # Shift right
            source_head_semantic = np.pad(source_head_semantic, ((0, 0), (horizontal_offset, 0)), mode='constant')[:, :-horizontal_offset]
            head_mask = np.pad(head_mask, ((0, 0), (horizontal_offset, 0)), mode='constant')[:, :-horizontal_offset]
        else:
            # Shift left
            source_head_semantic = np.pad(source_head_semantic, ((0, 0), (0, -horizontal_offset)), mode='constant')[:, -horizontal_offset:]
            head_mask = np.pad(head_mask, ((0, 0), (0, -horizontal_offset)), mode='constant')[:, -horizontal_offset:]
    
    # Convert masks to boolean arrays
    head_mask_bool = head_mask > 127
    body_mask_bool = body_mask > 127
    
    # Initialize blended layout with zeros
    blended_layout = np.zeros_like(source_head_semantic)
    
    # Copy semantic classes using boolean masks
    blended_layout[head_mask_bool] = source_head_semantic[head_mask_bool]
    blended_layout[body_mask_bool] = source_body_semantic[body_mask_bool]
    
    return blended_layout

# Set up directories
output_dir = 'outputs'
binary_masks_dir = os.path.join(output_dir, 'binary_masks')
blended_dir = os.path.join(output_dir, 'blended')
os.makedirs(blended_dir, exist_ok=True)

# Specify source files
head_source_path = os.path.join(output_dir, 'jensen.png')
body_source_path = os.path.join(output_dir, 'elon.png')
head_mask_path = os.path.join(binary_masks_dir, 'jensen_head_mask.png')
body_mask_path = os.path.join(binary_masks_dir, 'elon_body_mask.png')

# Load images and masks
head_source = np.array(Image.open(head_source_path))
body_source = np.array(Image.open(body_source_path))
head_mask = np.array(Image.open(head_mask_path))
body_mask = np.array(Image.open(body_mask_path))

# Print shapes for debugging
print("Shapes:")
print(f"head_source: {head_source.shape}")
print(f"body_source: {body_source.shape}")
print(f"head_mask: {head_mask.shape}")
print(f"body_mask: {body_mask.shape}")

# Blend layouts
blended = blend_semantic_layouts(head_source, body_source, head_mask, body_mask)

# Save original class indices (for later use)
np.save(os.path.join(blended_dir, 'blended_layout.npy'), blended)

# Scale values for visualization (0-19 to 0-255)
visualization = (blended * (255 // 19)).astype(np.uint8)

# Save both versions
# 1. Save visualization version
Image.fromarray(visualization).save(os.path.join(blended_dir, 'blended_layout_vis.png'))

# 2. Save original class indices version
Image.fromarray(blended.astype(np.uint8)).save(os.path.join(blended_dir, 'blended_layout.png'))

# Visualize results
plt.figure(figsize=(15, 3))
plt.subplot(151)
plt.imshow(head_source)
plt.title('Jensen Head')
plt.subplot(152)
plt.imshow(body_source)
plt.title('Elon Body')
plt.subplot(153)
plt.imshow(head_mask, cmap='gray')
plt.title('Jensen Head Mask')
plt.subplot(154)
plt.imshow(body_mask, cmap='gray')
plt.title('Elon Body Mask')
plt.subplot(155)
plt.imshow(blended)
plt.title('Blended Result')
plt.tight_layout()
plt.show()
     

In [ ]:
def blend_original_images(head_img, body_img, head_mask, body_mask, target_size=(480, 480)):
    """
    Blend original images using binary masks with alignment, with white background
    """
    # Resize all inputs to the same size
    head_img = cv2.resize(head_img, target_size[::-1])
    body_img = cv2.resize(body_img, target_size[::-1])
    head_mask = cv2.resize(head_mask, target_size[::-1], interpolation=cv2.INTER_NEAREST)
    body_mask = cv2.resize(body_mask, target_size[::-1], interpolation=cv2.INTER_NEAREST)
    
    # Find centers for alignment
    head_indices = np.where(head_mask > 127)
    body_indices = np.where(body_mask > 127)
    
    if len(head_indices[1]) > 0 and len(body_indices[1]) > 0:
        # Calculate and apply horizontal offset
        head_center = np.mean(head_indices[1])
        body_center = np.mean(body_indices[1])
        horizontal_offset = int(round(body_center - head_center))
        
        if horizontal_offset > 0:
            head_img = np.pad(head_img, ((0, 0), (horizontal_offset, 0), (0, 0)), mode='constant')[:, :-horizontal_offset]
            head_mask = np.pad(head_mask, ((0, 0), (horizontal_offset, 0)), mode='constant')[:, :-horizontal_offset]
        elif horizontal_offset < 0:
            head_img = np.pad(head_img, ((0, 0), (0, -horizontal_offset), (0, 0)), mode='constant')[:, -horizontal_offset:]
            head_mask = np.pad(head_mask, ((0, 0), (0, -horizontal_offset)), mode='constant')[:, -horizontal_offset:]
    
    # Create a mask for the original head in the body image
    original_head_mask = np.zeros_like(head_mask)
    original_head_indices = np.where(body_mask == 0)  # Areas where body mask is 0 are likely head
    if len(original_head_indices[0]) > 0:
        original_head_mask[original_head_indices] = 255
    
    # Start with a white background
    result = np.full_like(body_img, 255)  # Initialize with white (255)
    
    # Add body (excluding original head area)
    for i in range(3):
        # Add body where there's no original head
        result[:, :, i] = np.where(original_head_mask == 0, body_img[:, :, i], result[:, :, i])
        # Add new head
        result[:, :, i] = np.where(head_mask > 127, head_img[:, :, i], result[:, :, i])
    
    return result

# Load original images
head_orig = cv2.cvtColor(cv2.imread('inputs/jensen.png'), cv2.COLOR_BGR2RGB)
body_orig = cv2.cvtColor(cv2.imread('inputs/elon.jpg'), cv2.COLOR_BGR2RGB)

# Blend original images
final_result = blend_original_images(head_orig, body_orig, head_mask, body_mask)

# Save the result
plt.imsave(os.path.join(blended_dir, 'final_swap.png'), final_result)

# Visualize results
plt.figure(figsize=(15, 5))
plt.subplot(131)
plt.imshow(head_orig)
plt.title('Original Head (Jensen)')
plt.subplot(132)
plt.imshow(body_orig)
plt.title('Original Body (Elon)')
plt.subplot(133)
plt.imshow(final_result)
plt.title('Final Head Swap')
plt.tight_layout()
plt.show()